In [31]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pathlib import Path
import os

In [32]:
YELLOW_PATH = 'data/raw/yellow/2025/yellow_tripdata_2025-11.parquet'
ZONE_LOOKUP_PATH = 'data/taxi_zone_lookup.csv'
output_path = "/tmp/yellow_2025_11_repart4"

In [33]:
spark = (
    SparkSession.builder
    .appName("yellow-taxi-nov-2025-homework")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

In [34]:
# QUESTION 1
print("PySpark instaled & Spark Sesion created")
print("Spark version:", spark.version)


PySpark instaled & Spark Sesion created
Spark version: 3.5.8


In [35]:
df = spark.read.parquet(YELLOW_PATH)

print("\nSchema:")
df.printSchema()


Schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [36]:
# QUESTION 2
df.repartition(4).write.mode("overwrite").parquet(output_path)

parquet_files = list(Path(output_path).glob("*.parquet"))

sizes_mb = [f.stat().st_size / (1024 * 1024) for f in parquet_files]
avg_size_mb = sum(sizes_mb) / len(sizes_mb)

print("Num parquet files:", len(parquet_files))
print("Size (MB):", sizes_mb)
print("AVG Size (MB):", avg_size_mb)

[Stage 8:>                                                          (0 + 4) / 4]

Num parquet files: 4
Size (MB): [24.432312965393066, 24.403017044067383, 24.40389060974121, 24.424546241760254]
AVG Size (MB): 24.41594171524048


In [38]:
# QUESTION 3
df = df.withColumn(
    "pickup_ts",
    F.to_timestamp("tpep_pickup_datetime")
)

trips_15_nov = (
    df.filter(
        (F.col("pickup_ts") >= "2025-11-15") &
        (F.col("pickup_ts") < "2025-11-16")
    )
    .count()
)

print("Trips starting on 2025-11-15:", trips_15_nov)

Trips starting on 2025-11-15: 162604


In [40]:
# QUESTION 4
df_hours = df.withColumn(
    "trip_hours",
    (
        F.unix_timestamp("tpep_dropoff_datetime") -
        F.unix_timestamp("tpep_pickup_datetime")
    ) / 3600.0
)

longest_trip = df_hours.select(F.max("trip_hours")).collect()[0][0]

print("Longest trip (hours):", longest_trip)

Longest trip (hours): 90.64666666666666


In [44]:
# QUESTION 6
import urllib.request

url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
local_file = "taxi_zone_lookup.csv"

urllib.request.urlretrieve(url, local_file)

print("Download:", local_file)

Download: taxi_zone_lookup.csv


In [45]:
zone_df = (
    spark.read
    .option("header", "true")
    .csv("taxi_zone_lookup.csv")
)

zone_df.show(5)
zone_df.createOrReplaceTempView("zones")

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows



In [46]:
df.createOrReplaceTempView("yellow_trips")

spark.sql("""
SELECT
    z.Zone,
    COUNT(*) AS trips
FROM yellow_trips y
JOIN zones z
    ON y.PULocationID = z.LocationID
GROUP BY z.Zone
ORDER BY trips ASC
LIMIT 10
""").show(truncate=False)

+---------------------------------------------+-----+
|Zone                                         |trips|
+---------------------------------------------+-----+
|Governor's Island/Ellis Island/Liberty Island|1    |
|Eltingville/Annadale/Prince's Bay            |1    |
|Arden Heights                                |1    |
|Port Richmond                                |3    |
|Rikers Island                                |4    |
|Rossville/Woodrow                            |4    |
|Great Kills                                  |4    |
|Green-Wood Cemetery                          |4    |
|Jamaica Bay                                  |5    |
|Westerleigh                                  |12   |
+---------------------------------------------+-----+

